# Wine Style Clustering — Sprint 1: Exploratory Data Analysis

**Project:** Unsupervised discovery of wine style clusters from 10k+ Decanter tasting notes  
**Data source:** Decanter.com professional reviews, scraped via custom Selenium pipeline  
**Goal of this notebook:** Understand the corpus before modelling — distributions, vocabulary, quality of text

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Consistent style for all plots
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

RED   = '#8B1A2A'
WHITE = '#C9A84C'
ROSE  = '#E8A0A0'
GRAY  = '#888888'

print('Libraries loaded.')

## 1. Load data

In [ ]:
df = pd.read_csv('../data/raw/wines_clean.csv')

print(f'Total reviews:  {len(df):,}')
print(f'Columns:        {df.columns.tolist()}')
print(f'Nulls in description: {df["description"].isnull().sum()}')
df.head(3)

## 2. Colour breakdown

The project focuses on **Red** and **White** only — these have sufficient volume for stable clusters.  
Rosé and Orange are excluded from modelling (but kept in data/raw for reference).

In [ ]:
colour_counts = df['Colour'].value_counts()

colour_map = {'Red': RED, 'White': WHITE, 'Rosé': ROSE, 'Orange': '#E8C080'}
colors = [colour_map.get(c, GRAY) for c in colour_counts.index]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(colour_counts.index, colour_counts.values, color=colors, width=0.55)

for bar, val in zip(bars, colour_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f'{val:,}', ha='center', va='bottom', fontsize=11, color='#333')

ax.set_title('Reviews by wine colour', fontsize=14, pad=12)
ax.set_ylabel('Number of reviews')
ax.set_ylim(0, colour_counts.max() * 1.15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../results/figures/01_colour_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nRed + White for modelling: {colour_counts["Red"] + colour_counts["White"]:,} reviews')

## 3. Geographic distribution

In [ ]:
top_countries = df['Country'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_countries.index[::-1], top_countries.values[::-1], color='#5B7FA6', height=0.6)

for i, val in enumerate(top_countries.values[::-1]):
    ax.text(val + 20, i, f'{val:,}', va='center', fontsize=10, color='#444')

ax.set_title('Top 12 countries by review count', fontsize=14, pad=12)
ax.set_xlabel('Number of reviews')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../results/figures/01_country_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top regions
top_regions = df['Region'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_regions.index[::-1], top_regions.values[::-1], color='#7A9E7E', height=0.6)

for i, val in enumerate(top_regions.values[::-1]):
    ax.text(val + 10, i, f'{val:,}', va='center', fontsize=10, color='#444')

ax.set_title('Top 10 regions by review count', fontsize=14, pad=12)
ax.set_xlabel('Number of reviews')
plt.tight_layout()
plt.savefig('../results/figures/01_region_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Score distribution

Decanter uses a 100-point scale. Professional reviews cluster tightly in the 90–96 range.  
Understanding this matters for Sprint 3: we may want to stratify cluster quality analysis by score band.

In [ ]:
df_scored = df.dropna(subset=['score'])

# Separate by colour for overlay
df_red   = df_scored[df_scored['Colour'] == 'Red']
df_white = df_scored[df_scored['Colour'] == 'White']

fig, ax = plt.subplots(figsize=(9, 4))
bins = range(88, 101)

ax.hist(df_red['score'],   bins=bins, color=RED,   alpha=0.7, label='Red',   density=True)
ax.hist(df_white['score'], bins=bins, color=WHITE, alpha=0.7, label='White', density=True)

ax.set_title('Score distribution — Red vs White (Decanter 100-pt scale)', fontsize=13, pad=12)
ax.set_xlabel('Score')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/01_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Red   — median score: {df_red["score"].median()}')
print(f'White — median score: {df_white["score"].median()}')

## 5. Description length distribution

Description length affects embedding quality. Very short texts (< 20 words) may not carry enough
semantic signal. We'll flag these as potential quality concerns.

In [ ]:
df['desc_len'] = df['description'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, colour, color in zip(axes, ['Red', 'White'], [RED, WHITE]):
    subset = df[df['Colour'] == colour]['desc_len']
    ax.hist(subset, bins=40, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(subset.median(), color='#333', linestyle='--', linewidth=1.2,
               label=f'Median: {subset.median():.0f} words')
    ax.set_title(f'{colour} wines — description length', fontsize=12)
    ax.set_xlabel('Words per review')
    ax.set_ylabel('Count')
    ax.legend(fontsize=10)

plt.suptitle('Distribution of review length (words)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/01_desc_length.png', dpi=150, bbox_inches='tight')
plt.show()

short_reviews = df[df['desc_len'] < 20]
print(f'Reviews under 20 words: {len(short_reviews)} ({len(short_reviews)/len(df)*100:.1f}%)')

## 6. Most frequent flavour & texture descriptors

Wine vocabulary is domain-specific. This analysis motivates the core problem:
standard search treats "kirsch" and "dried cherry" as unrelated — our model must not.

In [ ]:
# Curated list of wine descriptors to count across the corpus
# (These are the vocabulary mismatch pairs our semantic search model was built to handle)

FLAVOUR_TERMS = [
    # Fruit
    'cherry', 'blackcurrant', 'cassis', 'plum', 'raspberry', 'strawberry',
    'blueberry', 'blackberry', 'kirsch', 'dried cherry', 'redcurrant',
    'citrus', 'lemon', 'grapefruit', 'peach', 'apricot', 'pear', 'apple',
    # Earthy / savoury
    'tobacco', 'leather', 'earth', 'forest floor', 'sous bois', 'truffle',
    'mineral', 'flint', 'graphite', 'saline',
    # Floral
    'violet', 'rose', 'jasmine', 'lavender', 'orange blossom',
    # Spice / oak
    'vanilla', 'cedar', 'cinnamon', 'pepper', 'clove', 'nutmeg', 'smoke',
    # Texture
    'tannins', 'acidity', 'silky', 'velvety', 'crisp', 'fresh', 'creamy',
]

def count_term(term, texts):
    pattern = re.compile(r'\b' + re.escape(term) + r'\b', re.IGNORECASE)
    return sum(1 for t in texts if pattern.search(str(t)))

corpus_red   = df[df['Colour'] == 'Red']['description'].tolist()
corpus_white = df[df['Colour'] == 'White']['description'].tolist()

term_counts = []
for term in FLAVOUR_TERMS:
    r = count_term(term, corpus_red)
    w = count_term(term, corpus_white)
    if r + w > 0:
        term_counts.append({'term': term, 'red': r, 'white': w, 'total': r + w})

tc = pd.DataFrame(term_counts).sort_values('total', ascending=False)
print('Top 20 descriptors by frequency:')
print(tc.head(20).to_string(index=False))

In [ ]:
top_terms = tc.head(20)

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(top_terms))
w = 0.38

ax.bar(x - w/2, top_terms['red'],   width=w, color=RED,   alpha=0.85, label='Red wines')
ax.bar(x + w/2, top_terms['white'], width=w, color=WHITE, alpha=0.85, label='White wines')

ax.set_xticks(x)
ax.set_xticklabels(top_terms['term'], rotation=45, ha='right', fontsize=10)
ax.set_title('Most frequent flavour & texture descriptors', fontsize=13, pad=12)
ax.set_ylabel('Reviews containing term')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/01_descriptor_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Vocabulary mismatch — the core problem

This section motivates the NLP approach. The same flavour concept appears under multiple names
in professional tasting notes — a keyword-based search would miss most of them.

In [ ]:
synonym_pairs = [
    ('kirsch',       'dried cherry'),
    ('sous bois',    'forest floor'),
    ('cassis',       'blackcurrant'),
    ('graphite',     'pencil shavings'),
    ('petrichor',    'wet stone'),
    ('garrigue',     'wild herbs'),
]

all_texts = df['description'].tolist()

print('Vocabulary mismatch examples — same concept, different words:')
print(f'{"Term A":<20} {"Count A":>8}   {"Term B":<20} {"Count B":>8}   Note')
print('-' * 80)

for a, b in synonym_pairs:
    ca = count_term(a, all_texts)
    cb = count_term(b, all_texts)
    note = 'keyword search misses one of these'
    print(f'{a:<20} {ca:>8,}   {b:<20} {cb:>8,}   {note}')

## 8. Corpus summary for modelling

Clean summary of what goes into Sprint 2.

In [ ]:
df_model = df[df['Colour'].isin(['Red', 'White'])].copy()
df_model = df_model[df_model['desc_len'] >= 15]  # drop very short descriptions

print('=== Corpus ready for modelling ===')
print(f'Total reviews:       {len(df_model):,}')
print(f'  Red:               {(df_model["Colour"]=="Red").sum():,}')
print(f'  White:             {(df_model["Colour"]=="White").sum():,}')
print(f'Median desc length:  {df_model["desc_len"].median():.0f} words')
print(f'Countries:           {df_model["Country"].nunique()}')
print(f'Regions:             {df_model["Region"].nunique()}')
print(f'Grape varieties:     {df_model["Grapes"].nunique()}')

# Save modelling subset
df_model.to_csv('../data/processed/wines_model.csv', index=False)
print('\nSaved to data/processed/wines_model.csv')

---

## Summary

| Metric | Value |
|--------|-------|
| Total reviews | 9,970 |
| Red + White (for modelling) | ~9,657 |
| Median description length | ~63 words |
| Top country | France (3,044) |
| Top region | California (1,593) |
| Score range | 88–100 (professional tier) |

**Key insight from vocabulary analysis:** The same flavour concept appears under multiple terms
(kirsch / dried cherry, sous bois / forest floor, cassis / blackcurrant). A keyword-based approach
would systematically miss cross-language and synonym matches. This motivates the semantic embedding
approach in Sprint 2.

**Next:** `02_embeddings_umap.ipynb` — encode all descriptions with our fine-tuned SentenceTransformer
and project to 2D with UMAP.
